# 🧥 DiffFit-3D: Full 3D Virtual Try-On Training

Bu notebook ile tüm pipeline'ı çalıştırabilirsin:
1. GPU & Kütüphane kurulumu
2. Google Drive'dan veri bağlama
3. SMPL-X beden parametre çıkarımı
4. 3D giysi render
5. Model eğitimi

**Gereksinimler**: GPU runtime (T4/A100/H100)

---
## 1️⃣ GPU Kontrolü

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU bulunamadı! Runtime → Change runtime type → GPU seç')

---
## 2️⃣ Projeyi Clone Et

In [ ]:
import os

# === BURAYI DEĞİŞTİR: Kendi GitHub repo URL'in ===
REPO_URL = 'https://github.com/SerhanTelatar/DiffFit-3D.git'
PROJECT_DIR = '/content/DiffFit-3D'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print('✅ Repo cloned')
else:
    !cd {PROJECT_DIR} && git pull
    print('✅ Repo updated')

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

---
## 3️⃣ Kütüphaneleri Kur

In [ ]:
# Temel bağımlılıklar
!pip install -q omegaconf accelerate transformers diffusers
!pip install -q opencv-python-headless pillow scipy
!pip install -q lpips einops timm

# 3D Pipeline bağımlılıkları
!pip install -q smplx trimesh pyrender

# PyTorch3D (CUDA derleme gerekebilir)
import sys
import torch
pyt_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')

try:
    import pytorch3d
    print(f'✅ PyTorch3D zaten kurulu: {pytorch3d.__version__}')
except ImportError:
    print('PyTorch3D kuruluyor (birkaç dakika sürebilir)...')
    !pip install -q "git+https://github.com/facebookresearch/pytorch3d.git"
    print('✅ PyTorch3D kuruldu')

---
## 4️⃣ Google Drive'ı Bağla & Verileri Aç

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# === BURAYI DEĞİŞTİR: Drive'daki klasör yolun ===
DRIVE_DATA = '/content/drive/MyDrive/DiffFit-3D'

import os
print('Drive içeriği:')
for f in os.listdir(DRIVE_DATA):
    size = os.path.getsize(os.path.join(DRIVE_DATA, f)) / 1e6
    print(f'  {f} ({size:.1f} MB)')

In [ ]:
import zipfile
import shutil
from pathlib import Path

PROJECT = Path('/content/DiffFit-3D')
DRIVE = Path(DRIVE_DATA)

def extract_zip(zip_name, target_dir):
    """Drive'daki zip'i çıkar."""
    zip_path = DRIVE / zip_name
    if not zip_path.exists():
        print(f'  ⚠️ {zip_name} bulunamadı, atlıyorum')
        return
    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    print(f'  📦 {zip_name} açılıyor → {target_dir}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target)
    print(f'  ✅ {zip_name} tamamlandı')

# Dizin yapısını oluştur
for d in ['data/raw/images', 'data/processed/poses', 'data/processed/segments',
          'data/processed/densepose', 'data/processed/agnostic',
          'data/processed/smplx_params', 'data/processed/renders_3d',
          'data/processed/normal_maps', 'data/processed/depth_maps',
          'data/garments_3d', 'checkpoints/pretrained']:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)

print('📂 Zip dosyaları açılıyor...')
print()

# Kişi görüntüleri
extract_zip('images.zip', PROJECT / 'data/raw/images')

# Preprocessing çıktıları
extract_zip('poses.zip', PROJECT / 'data/processed')
extract_zip('segments.zip', PROJECT / 'data/processed')
extract_zip('densepose.zip', PROJECT / 'data/processed')
extract_zip('agnostic.zip', PROJECT / 'data/processed')

# 3D giysi mesh'leri
extract_zip('garments_3d.zip', PROJECT / 'data/garments_3d')

# SMPL-X + VPoser
extract_zip('pretrained.zip', PROJECT / 'checkpoints')

# CSV dosyalarını kopyala
for csv_name in ['train_pairs.csv', 'val_pairs.csv', 'test_pairs.csv']:
    src = DRIVE / csv_name
    if src.exists():
        shutil.copy2(src, PROJECT / 'data/raw' / csv_name)

print()
print('✅ Tüm veriler yüklendi!')

In [ ]:
# Veri kontrolü
from pathlib import Path
P = Path('/content/DiffFit-3D')

checks = {
    'Kişi Görüntüleri': len(list((P/'data/raw/images').glob('*.jpg'))),
    'Poz': len(list((P/'data/processed/poses').glob('*'))),
    'Segmentasyon': len(list((P/'data/processed/segments').glob('*'))),
    'DensePose': len(list((P/'data/processed/densepose').glob('*'))),
    'Agnostik': len(list((P/'data/processed/agnostic').glob('*'))),
    '3D Giysi (upper)': len(list((P/'data/garments_3d/upper_body').glob('*'))) if (P/'data/garments_3d/upper_body').exists() else 0,
    '3D Giysi (lower)': len(list((P/'data/garments_3d/lower_body').glob('*'))) if (P/'data/garments_3d/lower_body').exists() else 0,
    '3D Giysi (dress)': len(list((P/'data/garments_3d/dresses').glob('*'))) if (P/'data/garments_3d/dresses').exists() else 0,
    '3D Giysi (outer)': len(list((P/'data/garments_3d/outerwear').glob('*'))) if (P/'data/garments_3d/outerwear').exists() else 0,
    'SMPL-X Model': (P/'checkpoints/pretrained/smplx/SMPLX_NEUTRAL.npz').exists(),
    'train_pairs.csv': (P/'data/raw/train_pairs.csv').exists(),
}

print('📊 Veri Kontrolü:')
print('=' * 40)
for name, val in checks.items():
    status = '✅' if val else '❌'
    print(f'{status} {name}: {val}')

---
## 5️⃣ SMPL-X Beden Parametre Çıkarımı

Her kişi görüntüsünden 3D beden parametrelerini (şekil, poz) çıkarir.

In [ ]:
import sys
sys.path.insert(0, '/content/DiffFit-3D')

from src.data.preprocessing.extract_smplx import extract_smplx

extract_smplx(
    image_dir='data/raw/images',
    output_dir='data/processed/smplx_params',
    model_dir='checkpoints/pretrained/smplx',
    device='cuda',
    save_mesh=True,
    mesh_dir='data/processed/smplx_meshes',
)
print('✅ SMPL-X parametre çıkarımı tamamlandı!')

---
## 6️⃣ 3D Giysi Render

CLOTH3D mesh'lerini SMPL-X beden modellerine giydirip 2D'ye render eder.

In [ ]:
from src.data.preprocessing.render_garment import render_garments

render_garments(
    garments_dir='data/garments_3d',
    smplx_params_dir='data/processed/smplx_params',
    output_dir='data/processed/renders_3d',
    normal_maps_dir='data/processed/normal_maps',
    depth_maps_dir='data/processed/depth_maps',
    resolution=512,
    device='cuda',
)
print('✅ 3D giysi render tamamlandı!')

---
## 7️⃣ Eğitimi Başlat 🚀

In [ ]:
# Eğitim config'ini düzenle (16GB VRAM için)
TRAIN_CONFIG = {
    'batch_size': 2,
    'gradient_accumulation_steps': 8,  # efektif batch = 16
    'learning_rate': 1e-5,
    'num_epochs': 50,
    'mixed_precision': True,
    'gradient_checkpointing': True,
    'save_every': 5,  # Her 5 epoch'ta checkpoint kaydet
    'log_every': 100,  # Her 100 step'te log
}

print('Eğitim Konfigürasyonu:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# Eğitimi başlat
!python scripts/train.py \
    --config configs/train.yaml \
    --batch_size 2 \
    --gradient_accumulation 8 \
    --lr 1e-5 \
    --epochs 50 \
    --mixed_precision \
    --gradient_checkpointing \
    --save_dir checkpoints/runs

---
## 8️⃣ Checkpoint'ları Drive'a Kaydet

In [ ]:
import shutil
from pathlib import Path

# Checkpoint'ları Drive'a kopyala (Colab kapanırsa kaybolmasın)
src_ckpt = Path('checkpoints/runs')
dst_ckpt = Path(DRIVE_DATA) / 'checkpoints'
dst_ckpt.mkdir(parents=True, exist_ok=True)

if src_ckpt.exists():
    for f in src_ckpt.glob('*.pt'):
        shutil.copy2(f, dst_ckpt / f.name)
        print(f'✅ {f.name} → Drive kaydedildi')

# Preprocessing sonuçlarını da kaydet (tekrar çalıştırmamak için)
for folder in ['smplx_params', 'renders_3d', 'normal_maps', 'depth_maps']:
    src = Path(f'data/processed/{folder}')
    if src.exists() and any(src.iterdir()):
        dst = Path(DRIVE_DATA) / f'{folder}.zip'
        if not dst.exists():
            shutil.make_archive(str(dst).replace('.zip',''), 'zip', src)
            print(f'✅ {folder} → Drive kaydedildi')

print('\n✅ Tüm sonuçlar Google Drive\'a kaydedildi!')

---
## 9️⃣ Hızlı Test (Opsiyonel)

Eğitilmiş model ile tek bir görüntüde try-on test et.

In [ ]:
from src.inference.image_tryon import ImageTryOn
from PIL import Image
import matplotlib.pyplot as plt

# Son checkpoint'ı yükle
tryon = ImageTryOn(
    checkpoint_path='checkpoints/runs/latest.pt',
    device='cuda'
)

# Test görüntüleri
person_img = 'data/raw/images/00001_00.jpg'
garment_mesh = 'data/garments_3d/upper_body/00047_Top/mesh.obj'

result = tryon.run(person_img, garment_mesh)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(person_img)); axes[0].set_title('Kişi')
axes[1].imshow(result['render_3d']); axes[1].set_title('3D Render')
axes[2].imshow(result['output']); axes[2].set_title('Sonuç')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('test_result.png', dpi=150)
plt.show()